# Batching Token IDs

The tokenizer turns text into token IDs. For model training, we turn those token IDs into PyTorch tensors named `x` and `y`.

Each row in `x` is a context window. Each matching row in `y` is the same window shifted one token to the right, so the model learns next-token prediction.


## Colab Bootstrap

Run this cell first. If you already ran `colab_setup.ipynb` in the same Colab runtime, this will reuse that setup; if not, it installs the small dependency set and clones the repo.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/markschatza/llm-from-scratch.git"
REPO_REF = "codex/colab-setup-workflow"
REPO_DIR = Path("/content/llm-from-scratch")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pretrain" / "tokenizer.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repo root. Run this notebook from inside the llm-from-scratch checkout."
    )

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "numpy", "torch"], check=True)
    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
    project_root = REPO_DIR
else:
    project_root = find_project_root(Path.cwd())

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"project root: {project_root}")
print(f"repo ref: {REPO_REF if IN_COLAB else 'local checkout'}")
print(f"in colab: {IN_COLAB}")

## Imports


In [ ]:
from pathlib import Path

import torch

from pretrain.batching import BatchConfig, get_batch, load_token_ids, split_tokens
from pretrain.tokenizer import Tokenizer


## Create Or Load Token IDs

If the encoded `.bin` corpus does not exist yet, this cell trains the tokenizer and writes the token IDs first.


In [ ]:
CORPUS = Path("data/van-life-story.txt")
OUTPUT_DIR = Path("pretrain/data")
TOKEN_BIN = OUTPUT_DIR / "train.van-life-story.bin"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TOKEN_BIN.exists():
    tok = Tokenizer.train(CORPUS, vocab_size=1024, output_dir=OUTPUT_DIR, name="sp")
    all_tokens: list[int] = []
    with open(CORPUS, "r", encoding="utf-8") as fin:
        for line in fin:
            line = line.rstrip("\n")
            if line:
                all_tokens.extend(tok.encode(line))
    torch.tensor(all_tokens, dtype=torch.int64).numpy().astype("uint32").tofile(TOKEN_BIN)

print(f"token file: {TOKEN_BIN}")


## Load And Split


In [ ]:
tokens = load_token_ids(TOKEN_BIN)
train_tokens, val_tokens = split_tokens(tokens, train_fraction=0.9)

print(type(tokens))
print(tokens.dtype, tokens.shape)
print(f"train tokens: {len(train_tokens):,}")
print(f"val tokens: {len(val_tokens):,}")


## Sample A Batch

`x` and `y` are tensors with shape `(batch_size, context_length)`. For every row, `y` is `x` shifted one token forward.


In [ ]:
config = BatchConfig(batch_size=4, context_length=8, device="cpu")
generator = torch.Generator().manual_seed(123)

x, y = get_batch(train_tokens, config, generator=generator)

print(f"x shape: {tuple(x.shape)}")
print(f"y shape: {tuple(y.shape)}")
print("x[0]:", x[0].tolist())
print("y[0]:", y[0].tolist())
print("x[0][1:] == y[0][:-1]:", torch.equal(x[0, 1:], y[0, :-1]))


## Decode One Example

This makes the shift easier to see in text form.


In [ ]:
tok = Tokenizer.from_pretrained(OUTPUT_DIR / "sp.model")

print("x text:")
print(tok.decode(x[0].tolist()))
print()
print("y text:")
print(tok.decode(y[0].tolist()))